# Early Warning Signals: Bautin

Same EWS and EDMDc pipeline as the [saddle-node example](02a_rate_saddle.ipynb), using the **Bautin** rate-induced system from the bundled simulation code. State is 1D (first component of the 2D Bautin state); control is **λ(t)**. Hyperparameters (n_samples, num_timesteps, burnin, wL, ResDMD delay) are aligned with the saddle-node example for comparison.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kews" / "__init__.py").is_file())
NOTEBOOK_DIR = REPO_ROOT / "examples"
for path in (REPO_ROOT, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUTPUT_DIR = REPO_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from experiments.indicators import rolling_variance, rolling_acf, spectral_gap_series


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from experiments.systems import generate
import kews
from kews.observables import rff, time_delay, mlp

LABEL_FONTSIZE = 16
LEGEND_FONTSIZE = 14
TICK_FONTSIZE = 13


## Generate Trajectories

Generate tipping and non-tipping Bautin trajectories.

In [ ]:
n_samples = 50
num_timesteps = 1000
burnin_idx = 0

def radius_1d(traj_2d):
    """radius r(t) = sqrt(x^2+y^2)"""
    return np.sqrt(traj_2d[:, 0]**2 + traj_2d[:, 1]**2)

# Generate tipping trajectories (state + control λ(t) for each)
# Full: for EWS (window of num_timesteps ending at cutoff_idx) and for visualization.
tipping_trajectories = []
tipping_trajectories_viz = []
tipping_full_params = []
tipping_indices = []
key = jax.random.PRNGKey(42)

for i in range(n_samples):
    key, subkey = jax.random.split(key)
    full_result_tip, part_result, info = generate(
        subkey,
        tipping_type='r_bautin',
        null_simulation=False,
        num_timesteps=num_timesteps,
    )
    tipping_trajectories.append(radius_1d(part_result.trajectory[burnin_idx:]))
    tipping_trajectories_viz.append(radius_1d(full_result_tip.trajectory[burnin_idx:]))
    tipping_full_params.append(np.array(part_result.param_values[burnin_idx:], dtype=np.float64))
    tipping_indices.append(info.get('transition_index', None))

# Generate non-tipping trajectories (state + control for each)
non_tipping_trajectories = []
non_tipping_trajectories_viz = []
non_tipping_full_params = []

for i in range(n_samples):
    key, subkey = jax.random.split(key)
    full_result_nontip, part_result, info = generate(
        subkey,
        tipping_type='r_bautin',
        null_simulation=True,
        num_timesteps=num_timesteps,
    )
    non_tipping_trajectories.append(radius_1d(part_result.trajectory[burnin_idx:]))
    non_tipping_trajectories_viz.append(radius_1d(full_result_nontip.trajectory[burnin_idx:]))
    non_tipping_full_params.append(np.array(part_result.param_values[burnin_idx:], dtype=np.float64))

# Universal cutoff (earliest transition index) — for visualization only
cutoff_idx = min(idx - burnin_idx for idx in tipping_indices if idx is not None)
start_ews = max(0, cutoff_idx - num_timesteps)

# EWS evaluation: fixed-length window (last n_ews points) so EWS series length is consistent
tipping_controls = np.array([p for p in tipping_full_params])
# tipping_controls = (tipping_controls - tipping_controls.mean(axis=1, keepdims=True)) / (tipping_controls.std(axis=1, keepdims=True) + 1e-12)

non_tipping_controls = np.array([p for p in non_tipping_full_params])
# non_tipping_controls = (non_tipping_controls - non_tipping_controls.mean(axis=1, keepdims=True)) / (non_tipping_controls.std(axis=1, keepdims=True) + 1e-12)


## Visualize Trajectories

In [ ]:
# Plot trajectories (full trajectory, visualization only)
tip_arr = np.asarray(tipping_trajectories_viz)
nontip_arr = np.asarray(non_tipping_trajectories_viz)

x = np.arange(tip_arr.shape[1])

fig, ax = plt.subplots(2, 1, figsize=(6, 8), sharex=True)

# Tipping individual trajectories
for i in range(tip_arr.shape[0]):
    if i == 0:
        ax[0].plot(x, tip_arr[i], color="C3", lw=1, label='Tipping trajectories')

    ax[0].plot(x, tip_arr[i], color="C3", lw=1)

# Non-tipping envelope + mean
ax[0].fill_between(
    x,
    nontip_arr.min(axis=0),
    nontip_arr.max(axis=0),
    color="C0",
    alpha=0.5,
    linewidth=0,
    label="Non-tipping trajectories",
)
ax[0].plot(x, nontip_arr.mean(axis=0), color="C0", lw=3)

ax[0].axvline(cutoff_idx, color="0.4", linestyle="--", lw=2, label="Tipping point")
ax[0].set_ylabel("State", fontsize=LABEL_FONTSIZE)
ax[0].set_ylim(-1, 12)
ax[0].grid(alpha=0.2)
ax[0].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

ax[1].plot(full_result_tip.param_values[burnin_idx:], color="C3", lw=3, label=r"Tipping $\Lambda(t)$")
ax[1].plot(full_result_nontip.param_values[burnin_idx:], color="C0", lw=3, label=r"Non-tipping $\Lambda(t)$")
ax[1].set_xlabel("Time", fontsize=LABEL_FONTSIZE)
ax[1].set_ylabel("Parameter", fontsize=LABEL_FONTSIZE)
ax[1].grid(alpha=0.2)
ax[1].legend(frameon=False, fontsize=LEGEND_FONTSIZE)

for a in ax:
    a.spines["top"].set_visible(False)
    a.spines["right"].set_visible(False)
    a.tick_params(labelsize=TICK_FONTSIZE)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/r_bautin_traj.png", dpi=300, bbox_inches="tight")
plt.show()


## EWS Metrics

Same as the saddle-node example: rolling AR(1), variance, ResDMD (delay embedding), and ResDMD with control. Window length and ResDMD delay fixed to match saddle notebook.

In [ ]:
# EWS parameters
wL = 0.5  # Rolling window length (fraction of time series)
N_ENSEMBLES = 1 # Number of ensembles for the stochastic ResKMD runs



def compute_ews_metrics(trajectories, wL, controls=None, control_method="augmented", augmented_future_input="zero"):
    """Compute EWS metrics for a list of trajectories.

    If controls is provided, it must be a list of 1D arrays (one per trajectory),
    each of length cutoff_idx. ResDMD with control is then computed and returned
    as 'resdmd_control_series' and 'resdmd_control_mlp_series'.
    """
    ar1_series = []
    var_series = []
    resdmd_series = []
    spectral_gap_list = []
    resdmd_control_series = [] if controls is not None else None
    resdmd_control_mlp_series = [] if controls is not None else None

    for i, traj in enumerate(trajectories):
        data = np.asarray(traj)

        # AR1, variance, ResDMD
        ar1 = rolling_acf(data, window=wL)
        var = rolling_variance(data, window=wL)
        spectral_gap = spectral_gap_series(data, wL)

        ## Ensembling for stochastic ResKMD runs
        resdmd_ens = []
        resdmd_control_ens = [] if controls is not None else None
        resdmd_control_mlp_ens = [] if controls is not None else None
        if controls is not None:
            u = np.asarray(controls[i])

        for seed in range(N_ENSEMBLES):
            resdmd = kews.resdmd(data, window=wL, observable=time_delay(n_delay=20))
            resdmd_ens.append(np.array(resdmd))

            if controls is not None:
                resdmd_c = kews.resdmd(data, 
                    window=wL, control=u, 
                    observable=time_delay(n_delay=20), 
                    
                    control_method=control_method, future_input=augmented_future_input
                )
                
                resdmd_c_mlp = kews.resdmd(data, 
                    window=wL, control=u,
                    observable=mlp(hidden_layers=[8, 4], bottleneck_dim=2, seed=seed),
                    fit_kwargs=dict(
                        lr=1e-3,
                        epochs=1000,
                        batch_size=32,
                        shuffle=True
                    ),
                    control_method=control_method, future_input=augmented_future_input
                )
                resdmd_control_ens.append(np.array(resdmd_c))
                resdmd_control_mlp_ens.append(np.array(resdmd_c_mlp))

        ar1_series.append(np.array(ar1))
        var_series.append(np.array(var))
        resdmd_series.append(np.array(resdmd_ens).mean(axis=0))
        spectral_gap_list.append(spectral_gap)

        # ResDMD with control
        if controls is not None:
            resdmd_control_series.append(np.array(resdmd_control_ens).mean(axis=0))
            resdmd_control_mlp_series.append(np.array(resdmd_control_mlp_ens).mean(axis=0))

    out = {
        'ar1_series': ar1_series,
        'var_series': var_series,
        'resdmd_series': resdmd_series,
        'spectral_gap_series': spectral_gap_list,
    }
    if resdmd_control_series is not None:
        out['resdmd_control_series'] = resdmd_control_series
        out['resdmd_control_mlp_series'] = resdmd_control_mlp_series
    return out

# Compute EWS (trajectories are already window [start_ews:cutoff_idx] of length num_timesteps)
tipping_ews = compute_ews_metrics(tipping_trajectories, wL, controls=tipping_controls, control_method="augmented", augmented_future_input="future")
non_tipping_ews = compute_ews_metrics(non_tipping_trajectories, wL, controls=non_tipping_controls, control_method="augmented", augmented_future_input="future")


In [ ]:
# Plot EWS
start_idx = 0
t = burnin_idx + start_ews + np.arange(len(tipping_ews['ar1_series'][0]))

def masked_array(series_list, start_idx):
    arr = np.array(series_list, dtype=float)
    arr[:, :start_idx] = np.nan
    return arr

def plot_distribution(ax, series_list, color, label, start_idx, band_alpha=0.15):
    arr = masked_array(series_list, start_idx)
    median = np.nanmedian(arr, axis=0)
    q25, q75 = np.nanpercentile(arr, [25, 75], axis=0)
    ax.plot(t, median, color=color, label=label, linewidth=3)
    ax.fill_between(t, q25, q75, color=color, alpha=band_alpha, linewidth=0)

def robust_ylim(series_lists, start_idx, low=2, high=98, margin=0.05):
    """Compute ylim from percentile of valid values so end spikes don't dominate."""
    vals = np.concatenate([masked_array(s, start_idx)[:, start_idx:].ravel() for s in series_lists])
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return 0.0, 1.0
    y0, y1 = np.nanpercentile(vals, [low, high])
    if y0 == y1:
        y0, y1 = y0 - 0.5, y1 + 0.5
    pad = (y1 - y0) * margin
    return y0 - pad, y1 + pad

panel_specs = [
    ('spectral_gap_series', 'Spectral gap', False),
    ('ar1_series', 'AR1', False),
    ('var_series', 'Variance', False),
    ('resdmd_control_series', 'ResKMD with control (TDF)', True),
    ('resdmd_control_mlp_series', 'ResKMD with control (MLP)', True),
]

fig, axes = plt.subplots(1, len(panel_specs), figsize=(24, 4), sharex=True)

for i, (key, ylabel, use_robust_ylim) in enumerate(panel_specs):
    ax = axes[i]

    plot_distribution(ax, tipping_ews[key], "C3", "Tipping", start_idx)
    plot_distribution(ax, non_tipping_ews[key], "C0", "Non-tipping", start_idx)

    ax.set_xlabel("Time", fontsize=LABEL_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=LABEL_FONTSIZE)
    ax.set_xlim([t[0], t[-1]])
    ax.tick_params(labelsize=TICK_FONTSIZE)
    ax.grid(alpha=0.2)

    if use_robust_ylim:
        ax.set_ylim(robust_ylim([tipping_ews[key], non_tipping_ews[key]], start_idx))

    if i == 0:
        ax.legend(frameon=False, fontsize=LEGEND_FONTSIZE)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/r_bautin_ews.png', dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# AUROC benchmark across trajectories
import pandas as pd
from experiments.metrics import ensemble_roc, plot_auroc, plot_roc

y_true = np.r_[np.ones(len(tipping_ews['var_series'])), np.zeros(len(non_tipping_ews['var_series']))].astype(int)

result = ensemble_roc(
    y_true,
    {
        'AR1': tipping_ews['ar1_series'] + non_tipping_ews['ar1_series'],
        'Variance': tipping_ews['var_series'] + non_tipping_ews['var_series'],
        'ResKMDc (TDF)': tipping_ews['resdmd_control_series'] + non_tipping_ews['resdmd_control_series'],
        'ResKMDc (MLP)': tipping_ews['resdmd_control_mlp_series'] + non_tipping_ews['resdmd_control_mlp_series'],
    },
)

auroc_df = pd.DataFrame(
    [
        {
            'indicator': name,
            'auroc': curve.auroc,
            'n_valid': curve.n_valid,
            'n_missing': curve.n_missing,
        }
        for name, curve in result.curves.items()
    ]
).sort_values('auroc', ascending=False).reset_index(drop=True)
display(auroc_df)

fig, ax = plt.subplots(figsize=(6, 4))
plot_auroc(result, ax=ax, label_fontsize=LABEL_FONTSIZE, tick_labelsize=TICK_FONTSIZE)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/r_bautin_auroc.png', dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
plot_roc(
    result,
    ax=ax,
    title=None,
    chance=True,
    show_auroc=True,
    linewidth=3,
    bold_top_n=2,
    label_fontsize=LABEL_FONTSIZE,
    tick_labelsize=TICK_FONTSIZE,
    legend_fontsize=LEGEND_FONTSIZE,
)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/r_bautin_roc.png', dpi=300, bbox_inches='tight')
plt.show()
